# 1) importing  reletated libraries 

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 2) importing csv file

In [3]:
s_data = pd.read_excel("/Users/gautamkumarsah/Desktop/ML/spam.xlsx")

print("Shape:", s_data.shape)
print("Columns:", s_data.columns)
s_data.head()


Shape: (5575, 5)
Columns: Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# 3) Keep Only Required Columns (v1 = label, v2 = message)

In [4]:
s_data = s_data[['v1', 'v2']].copy()
s_data.columns = ['label', 'text']

s_data.head()




,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# 4) Clean Labels 

In [5]:
s_data['label'] = s_data['label'].astype(str).str.lower().str.strip()

# ham -> 0 , spam -> 1
s_data['label'] = s_data['label'].map({'ham': 0, 'spam': 1})

# remove rows where label or text is missing
s_data = s_data.dropna(subset=['label', 'text']).copy()

# convert label to int
s_data['label'] = s_data['label'].astype(int)

print(s_data['label'].value_counts())


label
0    4826
1     747
Name: count, dtype: int64


# 5) Clean Text (Simple Function)

In [6]:
# =========================
# 5) Clean Text (Simple Function)
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove links
    text = re.sub(r"[^a-z\s]", " ", text)         # keep only letters
    text = re.sub(r"\s+", " ", text).strip()      # remove extra spaces
    return text

s_data['clean_text'] = s_data['text'].apply(clean_text)
s_data.head()


,label,text,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...


# 6) Split Data into Train & Test

In [7]:
X = s_data['clean_text']
y = s_data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (4458,)
Test size: (1115,)


In [8]:
vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print("Vectorized Train shape:", X_train_vec.shape)
print("Vectorized Test shape:", X_test_vec.shape)



Vectorized Train shape: (4458, 6795)
Vectorized Test shape: (1115, 6795)


# 8) Train Model 

In [9]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

print("✅ Model trained successfully!")


✅ Model trained successfully!


# 9) Evaluate Model

In [10]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.9614349775784753

Confusion Matrix:
 [[966   0]
 [ 43 106]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.71      0.83       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.90      1115
weighted avg       0.96      0.96      0.96      1115



# 10) Predict on Custom Message

In [11]:
def predict_spam(msg):
    msg = clean_text(msg)
    msg_vec = vectorizer.transform([msg])
    pred = model.predict(msg_vec)[0]
    return "SPAM 🚫" if pred == 1 else "HAM ✅"

print("Prediction:",predict_spam("Oh k...i'm watching here:)"))
print("Prediction:",predict_spam("I HAVE A DATE ON SUNDAY WITH WILL!!"))

Prediction: HAM ✅
Prediction: HAM ✅
